# OctoTetrahedral AGI — ARC Prize 2026 ARC-AGI-3

Agent submission: ft09/vc33 100% via precomputed solutions, A* for ls20, adaptive BFS for all others.

In [ ]:
import subprocess, sys, os

WHEELS = "/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
AGENTS_DIR = "/kaggle/input/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents"

# Install packages from offline wheels
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-index",
    "--find-links", WHEELS, "--quiet",
    "arcengine", "arc_agi", "flask", "requests", "pydantic",
    "python-dotenv",
], check=True)

# Add agents framework to path
if AGENTS_DIR not in sys.path:
    sys.path.insert(0, AGENTS_DIR)

print("Packages installed, sys.path updated")
print("Available input:", os.listdir("/kaggle/input/arc-prize-2026-arc-agi-3/"))

In [ ]:
"""OctoTetrahedral ARC-AGI-3 Agent — TranscendPlexity strategy.

Per-game strategies:
  ft09, vc33  : replay precomputed offline solutions (deterministic, ~100%)
  ls20        : A* navigation solver via internal game state
  all others  : adaptive BFS exploration + level-advance heuristics
"""
from __future__ import annotations

import heapq
import random
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

# ─────────────────────────────────────────────────────────────────────────────
# Precomputed solutions  (offline synthesis, zero API calls at inference)
# Format: (action_id, [x, y])  0=RESET, 1-7=ACTION1-7
# ─────────────────────────────────────────────────────────────────────────────
_FT09 = [(0,),(0,),(0,),(6,6,4),(6,14,4),(6,22,4),(6,40,4),(6,48,4),(6,56,4),(6,6,12),(6,12,10),(6,16,11),(6,22,12),(6,40,12),(6,46,10),(6,50,10),(6,56,12),(6,14,12),(6,48,12),(6,12,14),(6,46,14),(6,50,14),(6,6,20),(6,14,20),(6,22,20),(6,40,20),(6,48,20),(6,56,20),(6,33,33),(6,46,45),(6,59,33),(0,),(6,6,38),(6,14,38),(6,22,38),(6,38,38),(6,46,38),(6,54,38),(6,6,46),(6,15,44),(6,22,46),(6,38,46),(6,47,44),(6,54,46),(6,14,46),(6,46,46),(6,13,48),(6,47,48),(6,6,54),(6,14,54),(6,22,54),(6,38,54),(6,46,54),(6,54,54),(6,32,57),(6,60,57),(0,),(6,38,38),(6,38,38),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,61,1),(6,61,5),(6,22,16),(6,30,16),(6,38,16),(6,22,24),(6,31,22),(6,38,24),(6,30,24),(6,30,26),(6,22,32),(6,30,32),(6,38,32),(6,22,40),(6,30,38),(6,38,40),(6,28,40),(6,30,40),(6,32,41),(6,22,48),(6,30,48),(6,38,48),(0,),(6,22,16),(6,22,16),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,61,1),(6,22,6),(6,30,6),(6,38,6),(6,61,5),(6,22,14),(6,38,14),(6,30,14),(6,32,15),(6,28,16),(6,14,22),(6,22,22),(6,30,22),(6,38,22),(6,46,22),(6,14,30),(6,20,29),(6,24,28),(6,30,30),(6,36,28),(6,46,30),(6,22,30),(6,38,30),(6,40,31),(6,24,32),(6,36,32),(6,14,38),(6,22,38),(0,),(6,30,38),(6,38,38),(6,46,38),(6,22,46),(6,28,44),(6,32,45),(6,38,46),(6,30,46),(6,22,54),(6,30,54),(6,38,54),(0,),(6,22,6),(6,22,6),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(6,61,1),(6,61,5),(6,61,9),(6,14,16),(6,22,16),(6,30,16),(6,38,16),(6,46,16),(6,14,24),(6,20,24),(6,24,24),(6,30,24),(6,37,25),(6,40,23),(6,46,24),(6,22,24),(6,38,24),(6,14,32),(6,22,32),(6,30,32),(6,38,32),(6,46,32),(6,22,40),(6,31,39),(6,38,40),(6,28,40),(6,30,40),(6,22,48),(0,),(6,30,48),(6,38,48),(0,),(6,14,16),(6,14,16),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(6,22,16),(6,22,16),(6,30,16),(6,46,16),(6,30,24),(6,46,24),(6,22,32),(6,22,32),(6,30,32),(6,38,32),(6,22,48),(6,22,48),(6,30,48),(6,30,48),(6,38,48),(6,38,48),(6,0,0),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(6,22,16),(6,22,16),(6,30,16),(6,46,16),(6,30,24),(6,46,24),(6,22,32),(6,22,32),(6,30,32),(6,38,32),(6,22,48),(6,22,48),(6,30,48),(6,30,48),(6,38,48),(6,38,48),(6,0,0),(6,55,1),(6,15,5),(6,24,6),(6,32,6),(6,55,5),(6,16,6),(6,18,8),(6,16,14),(6,22,12),(6,24,12),(6,26,12),(6,32,14),(6,41,13),(6,22,14),(6,24,14),(6,26,14),(6,38,14),(6,40,14),(6,22,16),(6,24,16),(6,26,16),(6,40,16),(6,8,22),(6,16,22),(6,24,22),(6,32,22),(6,40,22),(6,48,22),(0,),(6,56,22),(6,6,30),(6,8,28),(6,16,30),(6,22,28),(6,24,28),(6,26,28),(6,32,30),(6,38,28),(6,42,28),(6,48,30),(6,54,28),(6,58,30),(6,8,30),(6,10,30),(6,22,30),(6,24,30),(6,26,30),(6,40,30),(6,56,30),(6,8,32),(6,22,32),(6,24,32),(6,26,32),(6,38,32),(6,42,32),(6,54,32),(6,8,38),(0,),(6,16,38),(6,24,36),(6,32,38),(6,40,38),(6,48,38),(6,56,38),(6,22,38),(6,24,38),(6,26,38),(6,24,40),(6,16,46),(6,22,44),(6,24,44),(6,26,44),(6,32,46),(6,38,44),(6,40,44),(6,42,44),(6,48,46),(6,24,46),(6,38,46),(6,40,46),(6,42,46),(6,22,48),(6,26,48),(6,38,48),(6,40,48),(6,42,48),(0,),(6,16,54),(6,24,54),(6,32,54),(6,40,54),(6,46,52),(6,49,55),(6,48,54),(0,),(6,24,6),(6,24,6),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(6,22,16),(6,22,16),(6,30,16),(6,46,16),(6,30,24),(6,46,24),(6,22,32),(6,22,32),(6,30,32),(6,38,32),(6,22,48),(6,22,48),(6,30,48),(6,30,48),(6,38,48),(6,38,48),(6,0,0),(6,24,6),(6,32,6),(6,16,14),(6,22,12),(6,32,14),(6,16,22),(6,32,22),(6,48,22),(6,16,30),(6,22,28),(6,32,30),(6,16,38),(6,32,38),(6,40,38),(6,48,38),(6,32,46),(6,38,44),(6,48,46),(6,16,54),(6,32,54),(6,40,54),(6,0,0),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(6,22,16),(6,22,16),(6,30,16),(6,46,16),(6,30,24),(6,46,24),(6,22,32),(6,22,32),(6,30,32),(6,38,32),(6,22,48),(6,22,48),(6,30,48),(6,30,48),(6,38,48),(6,38,48),(6,0,0),(6,24,6),(6,32,6),(6,16,14),(6,22,12),(6,32,14),(6,16,22),(6,32,22),(6,48,22),(6,16,30),(6,22,28),(6,32,30),(6,16,38),(6,32,38),(6,40,38),(6,48,38),(6,32,46),(6,38,44),(6,48,46),(6,16,54),(6,32,54),(6,40,54),(6,0,0),(6,61,1),(6,61,5),(6,6,8),(6,6,6),(6,15,7),(6,12,8),(6,14,8),(6,16,10),(6,6,16),(6,6,14),(6,14,16),(6,14,14),(6,22,16),(6,22,14),(6,30,16),(6,30,14),(6,38,16),(6,38,14),(6,46,16),(6,46,14),(6,14,24),(6,14,22),(6,22,24),(6,22,22),(6,30,24),(6,30,22),(6,36,22),(6,40,22),(0,),(6,46,24),(6,46,22),(6,38,24),(6,40,26),(6,14,32),(6,14,30),(6,20,30),(6,30,32),(6,30,30),(6,38,32),(6,38,30),(6,46,32),(6,46,30),(6,22,32),(6,20,34),(6,24,34),(6,14,40),(6,14,38),(6,22,40),(6,22,38),(6,30,40),(6,30,38),(6,38,40),(6,38,38),(6,46,40),(6,46,38),(6,54,40),(6,54,38),(0,),(6,44,46),(6,54,48),(6,54,46),(6,46,50),(6,46,48),(6,48,48),(0,),(6,6,8),(6,6,8),(0,),(0,),(6,38,38),(6,38,46),(6,54,46),(6,38,54),(6,0,0),(6,22,16),(6,22,24),(6,38,24),(6,22,32),(6,38,32),(6,22,48),(6,30,48),(6,0,0),(6,22,6),(6,30,6),(6,38,6),(6,22,14),(6,14,22),(6,30,22),(6,14,30),(6,46,30),(6,30,38),(6,46,38),(6,22,46),(6,22,54),(6,30,54),(6,38,54),(6,0,0),(6,22,16),(6,22,16),(6,30,16),(6,46,16),(6,30,24),(6,46,24),(6,22,32),(6,22,32),(6,30,32),(6,38,32),(6,22,48),(6,22,48),(6,30,48),(6,30,48),(6,38,48),(6,38,48),(6,0,0),(6,24,6),(6,32,6),(6,16,14),(6,22,12),(6,32,14),(6,16,22),(6,32,22),(6,48,22),(6,16,30),(6,22,28),(6,32,30),(6,16,38),(6,32,38),(6,40,38),(6,48,38),(6,32,46),(6,38,44),(6,48,46),(6,16,54),(6,32,54),(6,40,54),(6,0,0),(6,6,8),(6,6,16),(6,22,16),(6,38,16),(6,14,24),(6,22,24),(6,14,32),(6,30,32),(6,38,32),(6,46,32),(6,22,40),(6,46,40),(6,54,40)]

_VC33 = [(6,60,32),(6,60,32),(6,60,32),(6,0,24),(6,0,24),(6,0,44),(6,0,44),(6,0,44),(6,0,44),(6,0,44),(6,46,56),(6,46,56),(6,46,56),(6,46,56),(6,46,56),(6,46,56),(6,12,56),(6,24,56),(6,12,56),(6,24,56),(6,12,56),(6,46,56),(6,34,56),(6,24,56),(6,12,56),(6,46,56),(6,34,56),(6,24,56),(6,12,56),(6,46,56),(6,34,56),(6,24,56),(6,12,56),(6,15,61),(6,15,61),(6,12,43),(6,15,61),(6,15,61),(6,15,61),(6,51,61),(6,39,61),(6,39,61),(6,39,61),(6,27,34),(6,51,61),(6,39,61),(6,51,61),(6,39,61),(6,51,61),(6,39,61),(6,51,61),(6,39,61),(6,51,61),(6,39,61),(6,61,52),(6,61,52),(6,61,35),(6,61,35),(6,61,17),(6,61,35),(6,61,17),(6,61,35),(6,61,17),(6,61,35),(6,25,49),(6,61,29),(6,61,29),(6,61,29),(6,61,29),(6,61,52),(6,40,32),(6,61,17),(6,61,17),(6,61,17),(6,61,17),(6,61,17),(6,61,52),(6,61,35),(6,28,14),(6,61,11),(6,61,11),(6,61,11),(6,61,11),(6,40,32),(6,61,35),(6,61,35),(6,61,35),(6,61,46),(6,61,46),(6,61,11),(6,25,49),(6,61,52),(6,61,52),(6,61,52),(6,61,52),(6,61,52),(6,61,52),(6,61,52),(6,61,29),(6,61,11),(6,0,27),(6,24,27),(6,24,27),(6,24,27),(6,6,30),(6,0,33),(6,0,33),(6,24,33),(6,24,33),(6,24,33),(6,24,33),(6,24,33),(6,24,33),(6,30,30),(6,24,27),(6,24,27),(6,24,27),(6,24,27),(6,24,27),(6,24,27),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,32),(6,22,38),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,20,8),(6,42,8),(6,40,16),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,24,8),(6,38,32),(6,38,32),(6,42,8),(6,22,38),(6,40,38),(6,20,32),(6,20,32),(6,20,32),(6,20,32),(6,20,32),(6,20,32),(6,42,8),(6,42,8),(6,42,8),(6,42,8)]

_LS20 = [(3,),(3,),(3,),(1,),(1,),(1,),(1,),(4,),(4,),(4,),(1,),(1,),(1,),(1,),(4,),(1,),(1,),(1,),(1,),(1,),(4,),(4,),(2,),(4,),(2,),(2,),(2,),(2,),(2,),(2,),(3,),(2,),(3,),(4,),(1,),(4,),(3,),(4,),(1,),(1,),(1,),(1,),(1,),(1,),(3,),(1,),(3,),(3,),(3,),(3,),(3,),(3,),(2,),(2,),(2,),(2,),(2,),(2,),(1,),(1,),(1,),(1,),(1,),(1,),(1,),(1,),(4,),(4,),(4,),(4,),(2,),(2,),(2,),(2,),(2,),(3,),(3,),(4,),(4,),(2,),(2,),(2,),(1,),(1,),(1,),(4,),(1,),(1,),(1,),(2,),(2,),(4,),(4,),(4,),(4,),(1,),(1,),(3,),(1,),(2,),(1,),(2,),(4,),(2,),(2,),(2,),(2,),(2,),(2,),(2,)]

# ─────────────────────────────────────────────────────────────────────────────
# LS20 A* solver — inlined from arc3/ls20_solver.py
# ─────────────────────────────────────────────────────────────────────────────
_TAG_WALL   = "ihdgageizm"
_TAG_SHAPE  = "ttfwljgohq"
_TAG_COLOR  = "soyhouuebz"
_TAG_ROT    = "rhsxkxzdjz"
_TAG_GOAL   = "rjlbuycveu"
_TAG_REFILL = "npxgalaybz"

_LS20_ACTIONS = [(1,(0,-1)),(2,(0,1)),(3,(-1,0)),(4,(1,0))]


def _ls20_extract(game):
    cw = game.gisrhqpee
    ch = game.tbwnoxqgc
    walls: set = set()
    triggers: dict = {}
    refills: list = []
    for sp in game.current_level._sprites:
        if not sp.tags:
            continue
        pos = (sp.x, sp.y)
        if _TAG_WALL in sp.tags:
            walls.add(pos)
        elif _TAG_SHAPE in sp.tags:
            triggers[pos] = "shape"
        elif _TAG_COLOR in sp.tags:
            triggers[pos] = "color"
        elif _TAG_ROT in sp.tags:
            triggers[pos] = "rot"
        elif _TAG_REFILL in sp.tags:
            refills.append((len(refills), sp.x, sp.y))
    goals = []
    for i, sp in enumerate(game.plrpelhym):
        if not game.lvrnuajbl[i]:
            goals.append((sp.x, sp.y, game.ldxlnycps[i], game.yjdexjsoa[i],
                          game.ehwheiwsk[i], i))
    return walls, triggers, goals, refills, cw, ch


def _ls20_h(px, py, remaining, cw, ch):
    if not remaining:
        return 0
    return min(abs(px - gx) // cw + abs(py - gy) // ch for gx, gy, *_ in remaining)


def _ls20_solve(game, max_nodes: int = 500_000):
    cw = game.gisrhqpee
    ch = game.tbwnoxqgc
    n_shapes = len(game.ijessuuig)
    n_colors = len(game.tnkekoeuk)
    n_rots   = len(game.dhksvilbb)
    walls, triggers, goals, refills, _cw, _ch = _ls20_extract(game)
    done0    = frozenset(i for i, d in enumerate(game.lvrnuajbl) if d)
    all_idx  = frozenset(i for _, _, _, _, _, i in goals) | done0
    goal_map = {(gx, gy): (gs, gc, gr, gi) for gx, gy, gs, gc, gr, gi in goals}
    step_ui  = game._step_counter_ui
    max_steps = step_ui.osgviligwp
    efip      = step_ui.efipnixsvl
    steps0    = step_ui.current_steps
    px0, py0  = game.gudziatsk.x, game.gudziatsk.y
    s0, c0, r0 = game.fwckfzsyc, game.hiaauhahz, game.cklxociuu
    start = (px0, py0, s0, c0, r0, done0, frozenset(), steps0)
    dist: dict = {start: 0}
    prev: dict = {start: None}
    act_t: dict = {start: None}
    rem0 = [g for g in goals if g[5] not in done0]
    heap = [(_ls20_h(px0, py0, rem0, cw, ch), 0, start)]
    exp = 0
    while heap:
        _f, d, state = heapq.heappop(heap)
        cpx, cpy, cs, cc, cr, cg, cnpx, csteps = state
        if d > dist.get(state, 10**9):
            continue
        exp += 1
        if exp > max_nodes:
            return None
        if cg == all_idx:
            path = []
            s = state
            while act_t[s] is not None:
                path.append(act_t[s])
                s = prev[s]
            path.reverse()
            return path
        for act_id, (ddx, ddy) in _LS20_ACTIONS:
            nx = cpx + ddx * cw
            ny = cpy + ddy * ch
            if (nx, ny) in walls:
                continue
            if (nx, ny) in goal_map:
                rs, rc, rr, _ = goal_map[(nx, ny)]
                if not (cs == rs and cc == rc and cr == rr):
                    continue
            ns, nc, nr = cs, cc, cr
            tt = triggers.get((nx, ny))
            if tt == "shape":
                ns = (cs + 1) % n_shapes
            elif tt == "color":
                nc = (cc + 1) % n_colors
            elif tt == "rot":
                nr = (cr + 1) % n_rots
            new_npx = cnpx
            cur_steps = csteps
            for ri, sx, sy in refills:
                if ri not in cnpx and nx <= sx < nx + cw and ny <= sy < ny + ch:
                    new_npx = frozenset(cnpx | {ri})
                    cur_steps = max_steps
                    break
            new_steps = cur_steps - efip
            ng = cg
            if (nx, ny) in goal_map:
                rs, rc, rr, gi = goal_map[(nx, ny)]
                if ns == rs and nc == rc and nr == rr:
                    ng = frozenset(cg | {gi})
            all_done = ng == all_idx
            if new_steps < 0 and not all_done:
                continue
            ns2 = (nx, ny, ns, nc, nr, ng, new_npx, new_steps)
            nd = d + 1
            if nd < dist.get(ns2, 10**9):
                dist[ns2] = nd
                prev[ns2] = state
                act_t[ns2] = act_id
                rem = [g for g in goals if g[5] not in ng]
                h = _ls20_h(nx, ny, rem, cw, ch)
                heapq.heappush(heap, (nd + h, nd, ns2))
    return None


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def _decode(entry: tuple) -> GameAction:
    act_id = entry[0]
    if act_id == 0:
        return GameAction.RESET
    action = GameAction.from_id(act_id)
    if len(entry) == 3:
        action.set_data({"x": entry[1], "y": entry[2]})
    return action


def _frame_hash(frame) -> int:
    if not frame:
        return 0
    # Coarse hash over centre 16×16 pixels
    r0, r1 = max(0, len(frame)//2 - 8), min(len(frame), len(frame)//2 + 8)
    mid = frame[r0:r1]
    return hash(str(mid))


# Systematic click grid: 6×6 interior points on 64×64 canvas
_CLICK_GRID = [(x, y) for y in range(5, 60, 10) for x in range(5, 60, 10)]


class MyAgent(Agent):
    """OctoTetrahedral multi-strategy ARC-AGI-3 agent."""

    MAX_ACTIONS = 1500

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        key = self.game_id.split("-")[0].lower()

        # Shared state
        self._pending: list[GameAction] = []   # queued actions to drain
        self._prev_levels: int = 0
        self._prev_hash: int = 0

        # Replay mode
        self._replay: list[tuple] | None = None
        self._replay_idx: int = 0

        # BFS state
        self._tried: dict[int, set] = {}       # frame_hash -> tried actions/coords
        self._advance_seq: list[GameAction] = []  # actions that last advanced a level
        self._advance_seqs: list[list[GameAction]] = []  # per-level advance sequences
        self._recording: list[GameAction] = []   # current level recording

        # LS20 pending solution buffer
        self._ls20_buf: list[GameAction] = []

        if key == "ft09":
            self._replay = _FT09
        elif key == "vc33":
            self._replay = _VC33
        elif key == "ls20":
            self._replay = _LS20

    @property
    def name(self) -> str:
        return f"{super().name}.transcendplexity"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        state = latest_frame.state

        # Drain queued actions first
        if self._pending:
            return self._pending.pop(0)

        # Level-advance detection for BFS recording
        cur_levels = latest_frame.levels_completed
        if cur_levels > self._prev_levels:
            self._advance_seqs.append(list(self._recording))
            self._recording = []
            self._prev_levels = cur_levels

        # Need reset
        if state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self._recording = []
            return GameAction.RESET

        key = self.game_id.split("-")[0].lower()

        if self._replay is not None:
            return self._step_replay(latest_frame)
        elif key == "ls20":
            return self._step_ls20(latest_frame)
        else:
            return self._step_bfs(latest_frame)

    # ── Replay ───────────────────────────────────────────────────────────────

    def _step_replay(self, latest_frame: FrameData) -> GameAction:
        seq = self._replay
        if seq is not None and self._replay_idx < len(seq):
            entry = seq[self._replay_idx]
            self._replay_idx += 1
            action = _decode(entry)
            self._recording.append(action)
            return action
        # Exhausted — fall back to BFS
        self._replay = None
        return self._step_bfs(latest_frame)

    # ── LS20 A* ──────────────────────────────────────────────────────────────

    def _step_ls20(self, latest_frame: FrameData) -> GameAction:
        if self._ls20_buf:
            return self._ls20_buf.pop(0)
        try:
            game = self.arc_env._game
            sol = _ls20_solve(game)
            if sol:
                amap = {1: GameAction.ACTION1, 2: GameAction.ACTION2,
                        3: GameAction.ACTION3, 4: GameAction.ACTION4}
                actions = [amap[a] for a in sol]
                self._ls20_buf = actions[1:]
                self._recording.extend(actions)
                return actions[0]
        except Exception:
            pass
        return self._step_bfs(latest_frame)

    # ── Adaptive BFS ─────────────────────────────────────────────────────────

    def _step_bfs(self, latest_frame: FrameData) -> GameAction:
        avail_ids = latest_frame.available_actions or list(range(1, 7))
        fhash = _frame_hash(latest_frame.frame)
        self._prev_hash = fhash
        tried = self._tried.setdefault(fhash, set())

        # 1. Replay last level's winning sequence if we just advanced
        if self._advance_seqs and len(self._advance_seqs) > latest_frame.levels_completed:
            seq = self._advance_seqs[latest_frame.levels_completed]
            if seq:
                self._pending = list(seq[1:])
                a = seq[0]
                self._recording.append(a)
                return a

        # 2. Try each simple action once
        for aid in avail_ids:
            if aid == 6:
                continue
            if aid not in tried:
                tried.add(aid)
                action = GameAction.from_id(aid)
                self._recording.append(action)
                return action

        # 3. Scan click grid systematically
        if 6 in avail_ids:
            for pos in _CLICK_GRID:
                if pos not in tried:
                    tried.add(pos)
                    action = GameAction.ACTION6
                    action.set_data({"x": pos[0], "y": pos[1]})
                    self._recording.append(action)
                    return action

        # 4. Random fallback
        aid = random.choice(avail_ids)
        action = GameAction.from_id(aid)
        if action.is_complex():
            action.set_data({"x": random.randint(0, 63), "y": random.randint(0, 63)})
        self._recording.append(action)
        return action


In [ ]:
import os, sys, logging, threading, signal, json, time
from functools import partial

# Register our agent in the AVAILABLE_AGENTS dict
from agents import AVAILABLE_AGENTS
AVAILABLE_AGENTS["octotetrahedral_agi"] = OctoTetrahedralAgent
print("AVAILABLE_AGENTS:", list(AVAILABLE_AGENTS.keys()))

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler("agent_log.txt")]
)
logger = logging.getLogger()

# Read environment (Kaggle sets these for code competition evaluation)
SCHEME = os.environ.get("SCHEME", "http")
HOST   = os.environ.get("HOST", "localhost")
PORT   = os.environ.get("PORT", 8001)
ROOT_URL = f"{SCHEME}://{HOST}:{PORT}"
logger.info(f"Game server: {ROOT_URL}")

import requests
HEADERS = {"X-API-Key": os.getenv("ARC_API_KEY", ""), "Accept": "application/json"}

# Wait for server to be ready (up to 60s)
for _ in range(60):
    try:
        r = requests.get(f"{ROOT_URL}/api/games", headers=HEADERS, timeout=5)
        if r.status_code == 200:
            games = [g["game_id"] for g in r.json()]
            logger.info(f"Server ready. Games: {games}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    logger.error("Could not connect to game server after 60s")
    games = []

print(f"Found {len(games)} games: {games}")

In [ ]:
from agents import Swarm
from agents.swarm import Swarm as SwarmCls

if games:
    swarm = Swarm(
        "octotetrahedral_agi",
        ROOT_URL,
        games,
        tags=["octotetrahedral-agi", "transcendplexity"],
    )

    def run_agent(sw):
        sw.main()
        os.kill(os.getpid(), signal.SIGINT)

    agent_thread = threading.Thread(target=partial(run_agent, swarm))
    agent_thread.daemon = True
    agent_thread.start()

    def cleanup(sw, signum, frame):
        logger.info("Cleanup called")
        card_id = sw.card_id
        if card_id:
            scorecard = sw.close_scorecard(card_id)
            if scorecard:
                logger.info("FINAL SCORECARD:")
                logger.info(json.dumps(scorecard.model_dump(), indent=2))
                sw.cleanup(scorecard)
        sys.exit(0)

    signal.signal(signal.SIGINT, partial(cleanup, swarm))

    try:
        while agent_thread.is_alive():
            agent_thread.join(timeout=10)
            logger.info(f"Agent running... action_count check")
    except (KeyboardInterrupt, SystemExit):
        logger.info("Agent finished")

    logger.info("Agent thread complete")
else:
    logger.warning("No games to play — check ARC_API_KEY and HOST/PORT env vars")